In [5]:
# ============================================
# TRAINING (OPTIMIZED)
# ============================================

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os
import cv2
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from tqdm import tqdm

# ===== CONFIG =====
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 16
NUM_FRAMES  = 16
IMG_SIZE    = 224
POSE_DIM    = 132
NUM_CLASSES = 10
POSE_CACHE  = "/kaggle/working/pose_cache"
VALID_EXTS  = (".mp4", ".avi")

# ============================================
# DATASET (NO MEDIAPIPE HERE)
# ============================================

class FastDataset(Dataset):
    def __init__(self, root, transform):
        self.samples   = []
        self.transform = transform

        if not os.path.exists(root):
            print(f"Warning: dataset root not found: {root}")
            return

        classes = sorted([
            c for c in os.listdir(root)
            if not c.startswith(".") and os.path.isdir(os.path.join(root, c))
        ])

        for cls_idx, cls in enumerate(classes):
            cls_dir = os.path.join(root, cls)
            for vid in os.listdir(cls_dir):
                if vid.startswith(".") or not vid.endswith(VALID_EXTS):
                    continue

                vid_path  = os.path.join(cls_dir, vid)
                # Match the prefixed save filename from precompute script
                split     = os.path.basename(root)   # e.g. "train"
                pose_file = f"{split}_{cls}_{vid}.npy"
                pose_path = os.path.join(POSE_CACHE, pose_file)

                if not os.path.exists(pose_path):
                    print(f"Missing pose cache, skipping: {pose_file}")
                    continue

                self.samples.append((vid_path, pose_path, cls_idx))

        print(f"Loaded {len(self.samples)} samples from {root}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        vid_path, pose_path, label = self.samples[idx]

        # ── Load frames ──────────────────────────
        cap   = cv2.VideoCapture(vid_path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        idxs  = np.linspace(0, max(total - 1, 1), NUM_FRAMES, dtype=int)

        frames = []
        for i in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = self.transform(frame)
            else:
                frame = torch.zeros(3, IMG_SIZE, IMG_SIZE)
            frames.append(frame)

        cap.release()

        frames = torch.stack(frames)                            # (T, 3, H, W)
        pose   = torch.from_numpy(np.load(pose_path)).float()  # (T, POSE_DIM)

        return frames, pose, label


# ============================================
# MODEL
# ============================================

class Model(nn.Module):
    def __init__(self):
        super().__init__()

        eff            = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.backbone  = nn.Sequential(*list(eff.children())[:-1])

        self.vis_proj  = nn.Linear(1280, 256)
        self.pose_proj = nn.Linear(POSE_DIM, 256)

        self.lstm      = nn.LSTM(512, 512, batch_first=True, bidirectional=True)

        self.cls       = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(1024, NUM_CLASSES)
        )

    def forward(self, x, pose):
        B, T, C, H, W = x.shape

        # Visual features
        x    = x.view(B * T, C, H, W)
        feat = self.backbone(x).view(B, T, 1280)

        v = self.vis_proj(feat)   # (B, T, 256)
        p = self.pose_proj(pose)  # (B, T, 256)

        x = torch.cat([v, p], dim=-1)   # (B, T, 512)
        x, _ = self.lstm(x)             # (B, T, 1024)

        x = x.mean(dim=1)               # (B, 1024)
        return self.cls(x)


# ============================================
# TRANSFORMS
# ============================================

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ============================================
# DATALOADERS
# ============================================

train_ds = FastDataset("/kaggle/input/datasets/yuganjain/cricket-shot/cricket-shot/cricketshot/cricketshot/train", train_transform)
val_ds   = FastDataset("/kaggle/input/datasets/yuganjain/cricket-shot/cricket-shot/cricketshot/cricketshot/val",   val_transform)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

# ============================================
# TRAINING SETUP
# ============================================

model     = Model().to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
criterion = nn.CrossEntropyLoss()
scaler    = torch.cuda.amp.GradScaler()

best_val_acc = 0.0

# ============================================
# TRAIN LOOP
# ============================================

for epoch in range(30):

    # ── Train ──────────────────────────────
    model.train()
    total_loss    = 0
    correct_train = 0
    total_train   = 0

    for x, p, y in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]"):
        x = x.to(DEVICE, non_blocking=True)
        p = p.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            out  = model(x, p)
            loss = criterion(out, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss    += loss.item()
        preds          = out.argmax(dim=1)
        correct_train += (preds == y).sum().item()
        total_train   += y.size(0)

    train_acc = correct_train / total_train

    # ── Validate ────────────────────────────
    model.eval()
    correct_val = 0
    total_val   = 0

    with torch.no_grad():
        for x, p, y in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]"):
            x   = x.to(DEVICE, non_blocking=True)
            p   = p.to(DEVICE, non_blocking=True)
            y   = y.to(DEVICE, non_blocking=True)

            with torch.cuda.amp.autocast():
                out = model(x, p)

            preds       = out.argmax(dim=1)
            correct_val += (preds == y).sum().item()
            total_val   += y.size(0)

    val_acc = correct_val / total_val

    scheduler.step()

    print(
        f"Epoch {epoch+1:02d} | "
        f"Loss: {total_loss/len(train_loader):.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    # ── Save best model ─────────────────────
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "/kaggle/working/best_model.pth")
        print(f"  ✓ Saved best model (val_acc={val_acc:.4f})")

Loaded 1250 samples from /kaggle/input/datasets/yuganjain/cricket-shot/cricket-shot/cricketshot/cricketshot/train
Loaded 250 samples from /kaggle/input/datasets/yuganjain/cricket-shot/cricket-shot/cricketshot/cricketshot/val
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 147MB/s]
/tmp/ipykernel_55/3491970328.py:185: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler()
Epoch 1 [Train]:   0%|          | 0/79 [00:00<?, ?it/s]/tmp/ipykernel_55/3491970328.py:208: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1 [Val]:   0%|          | 0/16 [00:00<?, ?it/s]/tmp/ipykernel_55/3491970328.py:234: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1 [Val]: 100%|██████████| 16/16 [01:26<00:00,  5.40s/it]


Epoch 01 | Loss: 2.1517 | Train Acc: 0.2304 | Val Acc: 0.3560
  ✓ Saved best model (val_acc=0.3560)


Epoch 2 [Val]: 100%|██████████| 16/16 [01:23<00:00,  5.21s/it]


Epoch 02 | Loss: 1.5134 | Train Acc: 0.4792 | Val Acc: 0.5880
  ✓ Saved best model (val_acc=0.5880)


...

Epoch 13 | Loss: 0.1669 | Train Acc: 0.9752 | Val Acc: 0.8080
  ✓ Saved best model (val_acc=0.8080)


Epoch 14 [Val]: 100%|██████████| 16/16 [01:23<00:00,  5.24s/it]


Epoch 14 | Loss: 0.1271 | Train Acc: 0.9768 | Val Acc: 0.7840


Epoch 15 [Train]:  20%|██        | 16/79 [01:50<07:15,  6.92s/it]


KeyboardInterrupt: 

In [4]:
# ============================================
# PRECOMPUTE POSE VECTORS — GPU ACCELERATED
# ============================================

import os
import cv2
import numpy as np
import urllib.request
from tqdm import tqdm
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# ── Paths ──────────────────────────────────
DATASET_PATH = "/kaggle/input/datasets/yuganjain/cricket-shot/cricket-shot/cricketshot/cricketshot"
SAVE_PATH    = "/kaggle/working/pose_cache"
MODEL_PATH   = "pose_landmarker_heavy.task"

os.makedirs(SAVE_PATH, exist_ok=True)

# ── Config ─────────────────────────────────
POSE_DIM      = 33 * 4
NUM_FRAMES    = 16
VALID_EXTS    = (".mp4", ".avi")

# Tune these to your GPU VRAM / CPU core count:
NUM_WORKERS   = 4    # parallel video-decode threads
BATCH_SIZE    = 8    # frames per detector batch (MediaPipe IMAGE mode = 1, but
                     # we parallelise across detector instances instead)

# ── Download model if missing ───────────────
if not os.path.exists(MODEL_PATH):
    print("Downloading pose_landmarker model...")
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
        "pose_landmarker_heavy/float16/latest/pose_landmarker_heavy.task",
        MODEL_PATH
    )
    print("Download complete.")

# ── GPU resize helper (falls back to CPU) ──
USE_CUDA_RESIZE = cv2.cuda.getCudaEnabledDeviceCount() > 0
if USE_CUDA_RESIZE:
    print(f"✓ CUDA resize enabled ({cv2.cuda.getCudaEnabledDeviceCount()} device(s))")
else:
    print("⚠  No CUDA device found — using CPU resize (inference still benefits from TF/GPU)")

def gpu_resize(frame, size=(256, 256)):
    """Resize on GPU if available, else CPU."""
    if USE_CUDA_RESIZE:
        gpu_frame = cv2.cuda_GpuMat()
        gpu_frame.upload(frame)
        gpu_resized = cv2.cuda.resize(gpu_frame, size)
        return gpu_resized.download()
    return cv2.resize(frame, size)

# ── Per-thread detector pool ────────────────
# MediaPipe detectors are NOT thread-safe; create one per worker thread.
_thread_local = threading.local()

def get_detector():
    """Return a thread-local PoseLandmarker instance."""
    if not hasattr(_thread_local, "detector"):
        base_options = python.BaseOptions(
            model_asset_path=MODEL_PATH,
            # Uncomment for GPU delegate (requires mediapipe[gpu] or compatible build):
            # delegate=python.BaseOptions.Delegate.GPU
        )
        options = vision.PoseLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.IMAGE,
            num_poses=1
        )
        _thread_local.detector = vision.PoseLandmarker.create_from_options(options)
    return _thread_local.detector

# ── Pose extraction ─────────────────────────
def extract_pose(frame):
    """Extract a 132-d pose vector from a single RGB frame."""
    # Optional: resize to fixed resolution to speed up landmark detection
    frame = gpu_resize(frame, (256, 256))
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
    result = get_detector().detect(mp_image)
    if result.pose_landmarks:
        vec = np.array(
            [[lm.x, lm.y, lm.z, lm.visibility] for lm in result.pose_landmarks[0]],
            dtype=np.float32
        ).flatten()
        return vec
    return np.zeros(POSE_DIM, dtype=np.float32)

# ── Batch pose extraction for a list of frames
def extract_poses_batch(frames):
    """Run pose detection on a list of frames; returns list of vectors."""
    return [extract_pose(f) for f in frames]

# ── Video decoding — decode all target frames at once ──
def decode_frames(path):
    """
    Decode NUM_FRAMES evenly-spaced frames from a video.
    Uses cv2.CAP_PROP_POS_FRAMES for random access.
    Returns a list of RGB uint8 arrays (or zero arrays on failure).
    """
    cap   = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs  = np.linspace(0, max(total - 1, 1), NUM_FRAMES, dtype=int)
    frames = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        else:
            frames.append(None)
    cap.release()
    return frames

# ── Single-video pipeline (decode → pose → save) ──
def process_video(path, save_file):
    """Full pipeline for one video. Called from thread pool."""
    frames = decode_frames(path)
    poses  = []
    for frame in frames:
        if frame is not None:
            poses.append(extract_pose(frame))
        else:
            poses.append(np.zeros(POSE_DIM, dtype=np.float32))
    result = np.stack(poses)          # (NUM_FRAMES, POSE_DIM)
    np.save(save_file, result)
    return True

# ── Collect all jobs ────────────────────────
jobs = []  # list of (path, save_file)

for split in ["train", "val", "test"]:
    split_dir = os.path.join(DATASET_PATH, split)
    if not os.path.exists(split_dir):
        print(f"Skipping missing split: {split}")
        continue
    for cls in os.listdir(split_dir):
        if cls.startswith("."):
            continue
        cls_dir = os.path.join(split_dir, cls)
        if not os.path.isdir(cls_dir):
            continue
        for vid in os.listdir(cls_dir):
            if vid.startswith(".") or not vid.endswith(VALID_EXTS):
                continue
            path      = os.path.join(cls_dir, vid)
            save_file = os.path.join(SAVE_PATH, f"{split}_{cls}_{vid}.npy")
            if os.path.exists(save_file):   # already done
                continue
            jobs.append((path, save_file))

print(f"Total videos to process: {len(jobs)}")

# ── Parallel execution ──────────────────────
errors = []

with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    future_to_job = {
        executor.submit(process_video, path, save_file): (path, save_file)
        for path, save_file in jobs
    }
    with tqdm(total=len(jobs), desc="Processing videos") as pbar:
        for future in as_completed(future_to_job):
            path, _ = future_to_job[future]
            try:
                future.result()
            except Exception as e:
                errors.append((path, str(e)))
                tqdm.write(f"Failed: {path} — {e}")
            pbar.update(1)

# ── Summary ─────────────────────────────────
print(f"\nDone. Poses saved to: {SAVE_PATH}")
print(f"Successful: {len(jobs) - len(errors)} / {len(jobs)}")
if errors:
    print(f"Failed ({len(errors)}):")
    for p, e in errors:
        print(f"  {p}: {e}")

2026-04-17 13:50:31.396848: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776433831.619221      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776433831.678935      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776433832.220445      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776433832.220485      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776433832.220488      55 computation_placer.cc:177] computation placer alr

Download complete.
⚠  No CUDA device found — using CPU resize (inference still benefits from TF/GPU)
Total videos to process: 1750


Processing videos:   0%|          | 0/1750 [00:00<?, ?it/s]INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1776433853.215826     174 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776433853.233034     179 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776433853.246664     185 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776433853.379351     189 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776433853.399561     174 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling suppo


Done. Poses saved to: /kaggle/working/pose_cache
Successful: 1750 / 1750


In [2]:
!pip install mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 92.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 10.4 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [2]:
!pip install mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 103.6 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 10.6 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [3]:
import urllib.request

MODEL_PATH = "pose_landmarker.task"

# Download if not already present
if not os.path.exists(MODEL_PATH):
    print("Downloading pose_landmarker model...")
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/latest/pose_landmarker_heavy.task",
        MODEL_PATH
    )
    print("Download complete.")

Download complete.


In [20]:
# ============================================================
# BLOCK 2: REFERENCE EMBEDDING CREATION
# Loads trained weights from Block 1 — NO retraining
# Produces per-class embedding stores from 3–4 sample images
# ============================================================

import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ── Re-use exactly the same constants & classes from Block 1 ──
BEST_MODEL_PATH  = "/kaggle/input/datasets/yuganjain/modelsspec/best_model.pth"        # written by Block 1
EMBEDDINGS_PATH  = "reference_embeddings.npz"

# Path to your reference dataset: subfolders named after each class,
# each containing 3–4 sample images.
REFERENCE_PATH   = "/kaggle/input/datasets/yuganjain/modelimages/shot/shot"

POSE_DIM         = 33 * 4
EFFICIENTNET_DIM = 1280
IMG_SIZE         = 224

CLASSES = ['cover', 'defense', 'flick', 'hook', 'latecut',
           'lofted', 'pull', 'square_cut', 'straight', 'sweep']
NUM_CLASSES = len(CLASSES)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ============================================================
# COPY MODEL DEFINITION (must be identical to Block 1)
# ============================================================

class CricketModel(nn.Module):
    def __init__(self):
        super().__init__()
        eff            = efficientnet_b0(weights=None)
        self.backbone  = nn.Sequential(*list(eff.children())[:-1])
        self.vis_proj  = nn.Linear(1280, 256)
        self.pose_proj = nn.Linear(POSE_DIM, 256)          # single Linear, no Sequential
        self.lstm      = nn.LSTM(512, 512, batch_first=True, bidirectional=True)  # 1 layer
        self.cls       = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(1024, NUM_CLASSES)                    # no attn, no intermediate layer
        )

    def forward(self, x, pose):
        B, T, C, H, W = x.shape
        feat = self.backbone(x.view(B * T, C, H, W)).view(B, T, 1280)
        v    = self.vis_proj(feat)        # (B, T, 256)
        p    = self.pose_proj(pose)       # (B, T, 256)
        x    = torch.cat([v, p], dim=-1)  # (B, T, 512)
        x, _ = self.lstm(x)              # (B, T, 1024)
        x    = x.mean(dim=1)             # (B, 1024)  ← mean pooling, no attention
        return self.cls(x)

    def embed_frames(self, x, pose):
        """Returns per-frame LSTM outputs (B, T, 1024) for similarity scoring."""
        B, T, C, H, W = x.shape
        feat = self.backbone(x.view(B * T, C, H, W)).view(B, T, 1280)
        v    = self.vis_proj(feat)
        p    = self.pose_proj(pose)
        x    = torch.cat([v, p], dim=-1)
        x, _ = self.lstm(x)
        return x    # (B, T, 1024) — before mean pooling

# ============================================================
# LOAD TRAINED WEIGHTS  (the critical requirement)
# ============================================================

def load_trained_model(ckpt_path: str) -> CricketModel:
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")
    ckpt  = torch.load(ckpt_path, map_location=device)
    model = CricketModel().to(device)
    # Raw state dict (saved with torch.save(model.state_dict(), path))
    state_dict = ckpt['model_state_dict'] if isinstance(ckpt, dict) and 'model_state_dict' in ckpt else ckpt
    model.load_state_dict(state_dict)
    model.eval()
    print("Model loaded successfully.")
    return model

# ============================================================
# POSE EXTRACTOR  (same as Block 1)
# ============================================================

class PoseExtractor:
    def __init__(self):
        try:
            base_options = python.BaseOptions(model_asset_path="pose_landmarker.task")
            options = vision.PoseLandmarkerOptions(
                base_options=base_options,
                running_mode=vision.RunningMode.IMAGE,
                num_poses=1
            )
            self.detector = vision.PoseLandmarker.create_from_options(options)
        except Exception as e:
            print(f"Pose model failed: {e}")
            self.detector = None

    def extract(self, frame_rgb: np.ndarray) -> np.ndarray:
        if self.detector is None:
            return np.zeros(POSE_DIM, dtype=np.float32)
        try:
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
            result   = self.detector.detect(mp_image)
            if result.pose_landmarks:
                landmarks = result.pose_landmarks[0]
                vec = np.array(
                    [[lm.x, lm.y, lm.z, lm.visibility] for lm in landmarks],
                    dtype=np.float32
                ).flatten()
                lh, rh = landmarks[23], landmarks[24]
                cx, cy = (lh.x + rh.x) / 2, (lh.y + rh.y) / 2
                for i in range(33):
                    vec[i * 4]     -= cx
                    vec[i * 4 + 1] -= cy
                return vec
            return np.zeros(POSE_DIM, dtype=np.float32)
        except Exception:
            return np.zeros(POSE_DIM, dtype=np.float32)

# ============================================================
# IMAGE → EMBEDDING
# ============================================================

image_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


def image_to_embedding(model, pose_extractor, image_path):
    frame_bgr = cv2.imread(image_path)
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    
    pose_vec = pose_extractor.extract(frame_rgb)
    img_t    = image_tf(frame_rgb).unsqueeze(0).unsqueeze(0).to(device)  # (1,1,3,H,W)
    pose_t   = torch.from_numpy(pose_vec).unsqueeze(0).unsqueeze(0).to(device)  # (1,1,POSE_DIM)

    with torch.no_grad():
        B, T, C, H, W = img_t.shape
        feat = model.backbone(img_t.view(B*T, C, H, W)).view(B, T, 1280)
        v    = model.vis_proj(feat)          # (1, 1, 256)
        p    = model.pose_proj(pose_t)       # (1, 1, 256)
        emb  = torch.cat([v, p], dim=-1)     # (1, 1, 512)  ← stop here

    return emb.squeeze().cpu().numpy()       # (512,)
# ============================================================
# BUILD REFERENCE EMBEDDINGS
# ============================================================

def build_reference_embeddings(model, pose_extractor, ref_root):
    """
    Walk ref_root/<class_name>/<image_files> and build a dict:
        { class_name: np.ndarray of shape (N, 1024) }
    """
    class_embeddings = {}

    for cls in CLASSES:
        cls_dir = os.path.join(ref_root, cls)
        if not os.path.exists(cls_dir):
            print(f"  [SKIP] Class directory not found: {cls_dir}")
            continue

        image_files = [
            f for f in sorted(os.listdir(cls_dir))
            if (not f.startswith('.') and
                f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')))
        ]

        if not image_files:
            print(f"  [SKIP] No images found for class '{cls}'")
            continue

        print(f"  Processing class '{cls}' — {len(image_files)} image(s)")
        embeddings = []
        for fname in tqdm(image_files, desc=f"    {cls}", leave=False):
            path = os.path.join(cls_dir, fname)
            emb  = image_to_embedding(model, pose_extractor, path)
            embeddings.append(emb)

        class_embeddings[cls] = np.stack(embeddings)  # (N, 1024)
        print(f"    → {class_embeddings[cls].shape}")

    return class_embeddings


def save_embeddings(class_embeddings: dict, save_path: str):
    """Save class-wise embeddings as a compressed .npz file."""
    np.savez_compressed(save_path, **class_embeddings)
    print(f"\nReference embeddings saved to: {save_path}")
    print(f"Classes stored: {list(class_embeddings.keys())}")
    for cls, arr in class_embeddings.items():
        print(f"  {cls}: {arr.shape}")


def load_embeddings(path: str) -> dict:
    """Load reference embeddings back as a plain dict."""
    data = np.load(path, allow_pickle=False)
    return {k: data[k] for k in data.files}

# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    print("\n" + "=" * 60)
    print("  BLOCK 2 — Reference Embedding Creation")
    print("=" * 60 + "\n")

    model          = load_trained_model(BEST_MODEL_PATH)
    pose_extractor = PoseExtractor()

    print(f"\nBuilding embeddings from: {REFERENCE_PATH}\n")
    class_embeddings = build_reference_embeddings(model, pose_extractor, REFERENCE_PATH)

    if not class_embeddings:
        raise RuntimeError("No embeddings created. Check REFERENCE_PATH and folder structure.")

    save_embeddings(class_embeddings, EMBEDDINGS_PATH)
    print("\nBlock 2 complete.")

Using device: cuda

  BLOCK 2 — Reference Embedding Creation



W0000 00:00:1776428964.890265     299 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776428964.994073     299 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Model loaded successfully.

Building embeddings from: /kaggle/input/datasets/yuganjain/modelimages/shot/shot

  Processing class 'cover' — 4 image(s)


    → (4, 512)
  Processing class 'defense' — 4 image(s)


    → (4, 512)
  Processing class 'flick' — 4 image(s)


...

    → (3, 512)
  Processing class 'pull' — 3 image(s)


    → (3, 512)
  Processing class 'square_cut' — 4 image(s)


    → (4, 512)
  Processing class 'straight' — 4 image(s)


    → (4, 512)
  Processing class 'sweep' — 3 image(s)


    → (3, 512)

Reference embeddings saved to: reference_embeddings.npz
Classes stored: ['cover', 'defense', 'flick', 'hook', 'lofted', 'pull', 'square_cut', 'straight', 'sweep']
  cover: (4, 512)
  defense: (4, 512)
  flick: (4, 512)
  hook: (3, 512)
  lofted: (3, 512)
  pull: (3, 512)
  square_cut: (4, 512)
  straight: (4, 512)
  sweep: (3, 512)

Block 2 complete.


In [23]:
# ============================================================
# BLOCK 3: INFERENCE + SIMILARITY SCORING
# ============================================================

import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torchvision.models import efficientnet_b0
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ── Shared constants ──
BEST_MODEL_PATH  = "/kaggle/input/datasets/yuganjain/modelsspec/best_model.pth"
EMBEDDINGS_PATH  = "/kaggle/working/reference_embeddings.npz"

POSE_DIM         = 33 * 4
EFFICIENTNET_DIM = 1280
IMG_SIZE         = 224
NUM_FRAMES       = 16

CLASSES = ['cover', 'defense', 'flick', 'hook', 'latecut',
           'lofted', 'pull', 'square_cut', 'straight', 'sweep']
NUM_CLASSES = len(CLASSES)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ============================================================
# MODEL DEFINITION
# ============================================================

class CricketModel(nn.Module):
    def __init__(self):
        super().__init__()
        eff            = efficientnet_b0(weights=None)
        self.backbone  = nn.Sequential(*list(eff.children())[:-1])
        self.vis_proj  = nn.Linear(1280, 256)
        self.pose_proj = nn.Linear(POSE_DIM, 256)
        self.lstm      = nn.LSTM(512, 512, batch_first=True, bidirectional=True)
        self.cls       = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(1024, NUM_CLASSES)
        )

    def forward(self, x, pose):
        B, T, C, H, W = x.shape
        feat = self.backbone(x.view(B * T, C, H, W)).view(B, T, 1280)
        v    = self.vis_proj(feat)
        p    = self.pose_proj(pose)
        x    = torch.cat([v, p], dim=-1)
        x, _ = self.lstm(x)
        x    = x.mean(dim=1)
        return self.cls(x)

    def embed_frames(self, x, pose):
        """
        Returns pre-LSTM per-frame embeddings as numpy array (T, 512).
        Stops before the LSTM so single-frame and multi-frame embeddings
        live in the same feature space — required for image↔video comparison.
        FIX 1: Added missing 'self'
        FIX 2: Stop before LSTM (512-d, not 1024-d)
        FIX 3: Returns numpy directly so caller needs no extra .cpu().numpy()
        """
        B, T, C, H, W = x.shape
        feat = self.backbone(x.view(B * T, C, H, W)).view(B, T, 1280)
        v    = self.vis_proj(feat)          # (B, T, 256)
        p    = self.pose_proj(pose)         # (B, T, 256)
        emb  = torch.cat([v, p], dim=-1)    # (B, T, 512)
        return emb.squeeze(0).cpu().numpy() # (T, 512)  — B=1 always here

# ============================================================
# POSE EXTRACTOR
# ============================================================

class PoseExtractor:
    def __init__(self):
        try:
            base_options = python.BaseOptions(model_asset_path="pose_landmarker.task")
            options = vision.PoseLandmarkerOptions(
                base_options=base_options,
                running_mode=vision.RunningMode.IMAGE,
                num_poses=1
            )
            self.detector = vision.PoseLandmarker.create_from_options(options)
        except Exception as e:
            print(f"Pose model failed: {e}")
            self.detector = None

    def extract(self, frame_rgb: np.ndarray) -> np.ndarray:
        if self.detector is None:
            return np.zeros(POSE_DIM, dtype=np.float32)
        try:
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
            result   = self.detector.detect(mp_image)
            if result.pose_landmarks:
                landmarks = result.pose_landmarks[0]
                vec = np.array(
                    [[lm.x, lm.y, lm.z, lm.visibility] for lm in landmarks],
                    dtype=np.float32
                ).flatten()
                lh, rh = landmarks[23], landmarks[24]
                cx, cy = (lh.x + rh.x) / 2, (lh.y + rh.y) / 2
                for i in range(33):
                    vec[i * 4]     -= cx
                    vec[i * 4 + 1] -= cy
                return vec
            return np.zeros(POSE_DIM, dtype=np.float32)
        except Exception:
            return np.zeros(POSE_DIM, dtype=np.float32)

# ============================================================
# UTILITIES
# ============================================================

video_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


def load_trained_model(ckpt_path: str) -> CricketModel:
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")
    ckpt  = torch.load(ckpt_path, map_location=device)
    model = CricketModel().to(device)
    state_dict = ckpt['model_state_dict'] if isinstance(ckpt, dict) and 'model_state_dict' in ckpt else ckpt
    model.load_state_dict(state_dict)
    model.eval()
    print("Model loaded successfully.")
    return model


def load_reference_embeddings(path: str) -> dict:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Embeddings file not found: {path}")
    data = np.load(path, allow_pickle=False)
    embs = {k: data[k] for k in data.files}
    print(f"Loaded reference embeddings for classes: {list(embs.keys())}")
    # Sanity check: warn if ref embeddings are 1024-d (built with old Block 2)
    for cls, arr in embs.items():
        if arr.shape[-1] != 512:
            print(f"  WARNING: '{cls}' embeddings are {arr.shape[-1]}-d, expected 512-d. "
                  "Re-run Block 2 with the pre-LSTM fix.")
    return embs


def sample_video_frames(video_path: str,
                         pose_extractor: PoseExtractor,
                         num_frames: int = NUM_FRAMES):
    """
    Sample `num_frames` uniformly from a video.
    Returns:
        frame_tensor : (1, T, 3, H, W)
        pose_tensor  : (1, T, POSE_DIM)
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs  = np.linspace(0, max(total - 1, 1), num_frames, dtype=int)

    frames, poses = [], []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if ret:
            rgb  = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pose = pose_extractor.extract(rgb)
            poses.append(torch.from_numpy(pose))
            frames.append(video_tf(rgb))
        else:
            frames.append(torch.zeros(3, IMG_SIZE, IMG_SIZE))
            poses.append(torch.zeros(POSE_DIM))

    cap.release()

    frame_tensor = torch.stack(frames).unsqueeze(0).to(device)  # (1, T, 3, H, W)
    pose_tensor  = torch.stack(poses).unsqueeze(0).to(device)   # (1, T, POSE_DIM)
    return frame_tensor, pose_tensor

# ============================================================
# SIMILARITY FUNCTIONS
# ============================================================

def cosine_similarity_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """a: (M, D), b: (N, D) → (M, N) pairwise cosine similarities."""
    a_norm = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-8)
    b_norm = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-8)
    return a_norm @ b_norm.T


def euclidean_distance_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """a: (M, D), b: (N, D) → (M, N) pairwise Euclidean distances."""
    diff = a[:, None, :] - b[None, :, :]
    return np.sqrt((diff ** 2).sum(axis=-1))


def aggregate_similarity(video_embs: np.ndarray,
                          ref_embs:   np.ndarray,
                          metric: str = 'cosine') -> float:
    """
    For each video frame, find its BEST matching reference embedding,
    then average those best-match scores across all frames.
    video_embs : (T, 512)
    ref_embs   : (N, 512)
    """
    if metric == 'cosine':
        sim_matrix     = cosine_similarity_matrix(video_embs, ref_embs)  # (T, N)
        best_per_frame = sim_matrix.max(axis=1)                           # (T,)
        return float(best_per_frame.mean())
    elif metric == 'euclidean':
        dist_matrix    = euclidean_distance_matrix(video_embs, ref_embs) # (T, N)
        best_per_frame = dist_matrix.min(axis=1)                          # (T,)
        return float(-best_per_frame.mean())   # negate: higher = more similar
    else:
        raise ValueError(f"Unknown metric '{metric}'. Choose 'cosine' or 'euclidean'.")

# ============================================================
# MAIN INFERENCE FUNCTION
# ============================================================

def classify_and_score(video_path:     str,
                        model:          CricketModel,
                        pose_extractor: PoseExtractor,
                        ref_embeddings: dict,
                        metric: str = 'cosine') -> dict:
    """
    Full inference pipeline for a single video.
    Steps:
      1. Sample frames from video
      2. Classify → predicted class + confidence
      3. Extract pre-LSTM per-frame embeddings  (T, 512)
      4. Compare against reference embeddings   (N, 512)
      5. Return best-match-averaged similarity score
    """
    print(f"\nProcessing: {os.path.basename(video_path)}")

    # ── STEP 1: Sample frames ──
    frame_tensor, pose_tensor = sample_video_frames(video_path, pose_extractor)

    # ── STEP 2: Classification (uses full forward pass with LSTM) ──
    with torch.no_grad():
        logits     = model(frame_tensor, pose_tensor)         # (1, NUM_CLASSES)
        probs      = torch.softmax(logits, dim=1).squeeze(0)  # (NUM_CLASSES,)
        class_idx  = int(probs.argmax().item())
        confidence = float(probs[class_idx].item())
        pred_class = CLASSES[class_idx]

    print(f"  Predicted class : {pred_class}  (confidence: {confidence:.3f})")

    # ── STEP 3: Pre-LSTM per-frame embeddings ──
    # FIX: embed_frames now returns numpy (T, 512) directly — no extra .cpu().numpy()
    with torch.no_grad():
        video_embs = model.embed_frames(frame_tensor, pose_tensor)  # (T, 512)

    print(f"  Video embeddings shape: {video_embs.shape}")

    # ── STEP 4: Similarity scoring ──
    if pred_class not in ref_embeddings:
        print(f"  WARNING: No reference embeddings found for '{pred_class}'. "
              "Score will be NaN.")
        sim_score = float('nan')
    else:
        ref_embs  = ref_embeddings[pred_class]  # (N, 512)
        sim_score = aggregate_similarity(video_embs, ref_embs, metric=metric)
        print(f"  Similarity score : {sim_score:.4f}  (metric: {metric})")

    return {
        'predicted_class':  pred_class,
        'class_index':      class_idx,
        'confidence':       confidence,
        'similarity_score': sim_score,
        'metric':           metric,
    }

# ============================================================
# BATCH INFERENCE
# ============================================================

def batch_infer(video_paths:    list,
                model:          CricketModel,
                pose_extractor: PoseExtractor,
                ref_embeddings: dict,
                metric: str = 'cosine') -> list[dict]:
    """Run inference on a list of video files and return a list of result dicts."""
    results = []
    for vp in tqdm(video_paths, desc="Inferring"):
        try:
            res = classify_and_score(vp, model, pose_extractor, ref_embeddings, metric)
            res['video'] = vp
        except Exception as e:
            print(f"  ERROR processing {vp}: {e}")
            res = {'video': vp, 'error': str(e)}
        results.append(res)
    return results

# ============================================================
# ENTRY POINT
# ============================================================

if __name__ == "__main__":
    print("\n" + "=" * 60)
    print("  BLOCK 3 — Inference + Similarity Scoring")
    print("=" * 60 + "\n")

    model          = load_trained_model(BEST_MODEL_PATH)
    ref_embeddings = load_reference_embeddings(EMBEDDINGS_PATH)
    pose_extractor = PoseExtractor()

    video_path = "/kaggle/input/datasets/yuganjain/cricket-shot/cricket-shot/cricketshot/cricketshot/test/pull/pull_0004.avi"

    result = classify_and_score(
        video_path     = video_path,
        model          = model,
        pose_extractor = pose_extractor,
        ref_embeddings = ref_embeddings,
        metric         = 'cosine',
    )

    print("\n" + "─" * 40)
    print("  FINAL RESULT")
    print("─" * 40)
    print(f"  Predicted class  : {result['predicted_class']}")
    print(f"  Confidence       : {result['confidence']:.4f}")
    print(f"  Similarity score : {result['similarity_score']:.4f}  ({result['metric']})")
    print("─" * 40)


  BLOCK 3 — Inference + Similarity Scoring

Model loaded successfully.
Loaded reference embeddings for classes: ['cover', 'defense', 'flick', 'hook', 'lofted', 'pull', 'square_cut', 'straight', 'sweep']


W0000 00:00:1776430409.541481     333 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776430409.651111     333 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.



Processing: pull_0004.avi
  Predicted class : pull  (confidence: 0.756)
  Video embeddings shape: (16, 512)
  Similarity score : 0.4859  (metric: cosine)

────────────────────────────────────────
  FINAL RESULT
────────────────────────────────────────
  Predicted class  : pull
  Confidence       : 0.7556
  Similarity score : 0.4859  (cosine)
────────────────────────────────────────


In [4]:
!pip install mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 88.7 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 10.3 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
